<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Multimodal - Bild
</b></font> </br></p>


---

<p><font color='darkblue' size="4">
ℹ️ <b>Hinweis</b>
</font></p>


> Dieses Notebook verwendet die **direkte OpenAI-API** für Bild-Endpunkte (GPT Image 2, ...), da LangChain keine nativen Wrapper für alle Bildgenerierungs- und Inpainting-Funktionen bereitstellt.


In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import IMAGE_GENERATION
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ HF_TOKEN erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.3.9
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.7
langchain-ollama                         1.1.0
langchain-openai                         1.3.3
langchain-protocol                       0.0.17
langchain-text-splitters                 1.1.2
langgraph                                1.2.5
langgraph-checkpoint                     4.1.1
langgraph-checkpoint-sqlite              3.1.0
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.2

IP-Adresse: 34.87.30.209
Hostname: 209.30.87.34.bc.googleusercontent.com
Stadt: Singapore
Region: Singapore
Land: SG
Koordinaten: 1.2897,103.8501
Provi

In [2]:
#@title 🛠️ Installationen { display-mode: "form" }


In [3]:
#@title 📂 Dokumente, Bilder { display-mode: "form" }

# ── Stdlib ────────────────────────────────────────────────────────────────────
import shutil

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import copy_from_github

shutil.rmtree("files", ignore_errors=True)

# --- Bilder
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="apfel.png",
)
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="peoples.png",
)
copy_from_github(
    source="ralf-42/GenAI/02_daten/02_bild",
    target="files",
    mask="hedra_cyborg*.png",
)

  Kopiere: 02_daten/02_bild/apfel.png → files/apfel.png

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/peoples.png → files/peoples.png

Ergebnis: 1 Datei(en) kopiert → files
  Kopiere: 02_daten/02_bild/hedra_cyborg.png → files/hedra_cyborg.png
  Kopiere: 02_daten/02_bild/hedra_cyborg_mask_image.png → files/hedra_cyborg_mask_image.png

Ergebnis: 2 Datei(en) kopiert → files


['files/hedra_cyborg.png', 'files/hedra_cyborg_mask_image.png']

In [4]:
#@markdown 🛠️ Hilfsfunktionen { display-mode: "form" }

# ── Stdlib ────────────────────────────────────────────────────────────────────
import base64
import os

# ── Web & APIs ────────────────────────────────────────────────────────────────
import requests

# ── Stdlib ────────────────────────────────────────────────────────────────────
from io import BytesIO

# ── Colab ─────────────────────────────────────────────────────────────────────
from google.colab import files
from IPython.display import Image as IPImage
from IPython.display import Markdown, display

# ── OpenAI ────────────────────────────────────────────────────────────────────
from openai import OpenAI

# ── Multimodal ────────────────────────────────────────────────────────────────
from PIL import Image as PILImage   # ImageDraw nur lokal dort importieren, wo es gebraucht wird

# ── Machine Learning ──────────────────────────────────────────────────────────
from transformers import pipeline

IMAGE_DIR = "/content/files"

client = OpenAI()

def encode_image(image_path):
    """Kodiert ein Bild in Base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def show_img(img, width=512):
    """Zeigt ein PIL-Bild skaliert an."""
    ratio = width / img.width
    new_size = (width, int(img.height * ratio))
    display(img.resize(new_size))

def analyze_image(image_path, prompt, model="gpt-5.4-mini"):
    """Analysiert ein Bild mit einem Vision-Modell."""
    base64_image = encode_image(image_path)
    messages = [{
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
        ]
    }]
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)



# 1 | Intro
---

In [ ]:
#@markdown   <p><font size="4" color='green'>  Bild-Pipeline</font> </br></p>


# Multimodale Bild-Pipeline
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    INPUT([Prompt / Bild]) --> API[OpenAI API]
    API --> GEN[Bildgenerierung
GPT Image 2]
    API --> CLS[Klassifizierung
GPT-5.4-mini]
    API --> DESC[Bildbeschreibung
GPT-5.4-mini]
    API --> INP[Inpainting
GPT Image 2]
    API --> OP[Outpainting
GPT Image 2]

    GEN --> OUT([Ausgabe])
    CLS --> OUT
    DESC --> OUT
    INP --> OUT
    OP --> OUT
    style INPUT fill:#FF9800,color:#fff
    style OUT fill:#4CAF50,color:#fff
"""
mermaid(diagram, width=800)

Dieses Modul beschäftigt sich mit der Analyse von Bildern und ihrer Rolle als Basis für multimodale Systeme. Solche Modelle stellen einen bedeutenden Fortschritt in der KI dar, da sie geschriebene Sprache in visuelle Darstellungen umwandeln. Bekannte Beispiele wie DALL·E, Midjourney und Stable Diffusion nutzen Deep-Learning-Techniken, um detaillierte Bilder auf Grundlage von Textbeschreibungen zu erzeugen. Dabei erfassen sie die Zusammenhänge zwischen sprachlichen und visuellen Elementen, wodurch sie in der Lage sind, Szenen, Objekte und Konzepte entsprechend der gegebenen Beschreibung zu generieren.  

Multimodale Modelle gehen über diese Funktionalität hinaus, indem sie mehrere Datenquellen kombinieren – in der Regel Text,  Bilder, Audio oder Video. Ihr Ziel ist es, verschiedene Informationsarten simultan zu verarbeiten und in Beziehung zu setzen. So können sie beispielsweise Bildbeschreibungen aus Text erstellen, Fragen zu visuellen Inhalten beantworten oder sogar Videos auf Basis von Texteingaben generieren. Durch die Verbindung der Fähigkeiten von Text- und Bildmodellen erweitern multimodale Systeme das Anwendungsspektrum der KI und ermöglichen eine flexiblere, kontextbezogene Interaktion.  

Während **Text-zu-Bild-Modelle** primär darauf ausgerichtet sind, sprachliche Eingaben in visuelle Inhalte umzusetzen, ermöglichen **multimodale Modelle** eine komplexere Verknüpfung verschiedener Medientypen. In diesem Modul wird untersucht, wie diese Modelle funktionieren und wie ihre Architektur die Entwicklung weiterführender multimodaler Systeme beeinflusst. Zudem werden praxisnahe Anwendungsfälle betrachtet sowie mögliche zukünftige Entwicklungen aufgezeigt, die sich aus der Kombination unterschiedlicher Datenmodalitäten ergeben.

OpenAI bietet mit GPT Image 1/2 zwei fortschrittliche KI-Modelle zur Bildgenerierung an. Beide Modelle wandeln Texteingaben in Bilder um, unterscheiden sich jedoch in ihren Funktionen und Einsatzmöglichkeiten.



**Einsatzspektrum GPT Image 2:**



| Nr. | Aufgabe        | Kurzbeschreibung                                                  | Beispiel                                     | GPT Image 2             | GPT Image 2                    |
| --- | -------------- | ----------------------------------------------------------------- | -------------------------------------------- | ----------------------- | ------------------------------ |
| 1   | Text-zu-Bild   | Textbeschreibung → Bild; steuerbar nach Stil, Licht, Komposition. | Fuchs mit Kamera im impressionistischen Stil | ✅ Präzise & realistisch | ✅ Kreativ & hochwertig         |
| 2   | Inpainting     | Objekte entfernen oder ersetzen per Maske.                        | Person entfernen, Hintergrund auffüllen      | ✅ Voll über API         | ✅ Voll über API                |
| 3   | Outpainting    | Bild über Ränder erweitern.                                       | Panorama verbreitern                         | ✅ Präzise steuerbar     | ✅ Präzise steuerbar            |
| 4   | Bild-Variation | Gleiche Szene, anderer Stil.                                      | Nachtaufnahme im Cyberpunk-Stil              | ✅ Reproduzierbar        | ✅ Reproduzierbar               |
| 5   | Stil-Transfer  | Stiländerung über Prompt.                                         | Ölgemälde- oder Comic-Look                   | ✅ Kontrolliert          | ✅ Hochwertig & flexibel        |
| 6   | Compositing    | Elemente kombinieren oder reparieren.                             | Zwei Personen in ein Bild setzen             | ✅ Über Masken möglich   | ✅ Über Bild-Editierung möglich |



# 2 | Übersicht Bildverarbeitung
---

| **Kategorie**                        | **Aufgabe**                      | **Beschreibung**                                                                      |
| ------------------------------------ | -------------------------------- | ------------------------------------------------------------------------------------- |
| 🧠 **Analyse & Klassifikation**      | **Bildklassifikation**           | Zuordnung eines Bildes zu vordefinierten Klassen (z. B. Katze, Hund, Auto).           |
|                                      | **Objekterkennung**                  | Erkennung und Lokalisierung mehrerer Objekte in einem Bild mit Bounding Boxes.        |
|                                      | Bildsegmentierung                | Pixelgenaue Zuordnung von Bildbereichen zu Klassen (z. B. Straße, Baum, Mensch).      |
|                                      | Gesichtserkennung                | Identifikation oder Verifikation von Personen auf Bildern.                            |
|                                      | Emotionserkennung                | Analyse der Gesichtsausdrücke zur Einschätzung von Emotionen.                         |
|                                      | Anomalieerkennung                | Erkennung ungewöhnlicher oder fehlerhafter Bildinhalte (z. B. Produktionsfehler).     |
| 📝 **Generierung & Transformation**  | **Bildgenerierung**              | Erzeugung synthetischer Bilder, z. B. aus Text (Text-to-Image) oder Rauschen.         |
|                                      | Bild-zu-Bild-Übersetzung         | Umwandlung von Bildern (z. B. Skizze → Foto, Tag → Nacht).                            |
|                                      | Stiltransfer                     | Übertragung des Stils eines Bildes auf ein anderes (z. B. Van Gogh → Foto).           |
|                                      | Super-Resolution                 | Hochskalierung von Bildern mit künstlicher Detailschärfung.                           |
|                                      | Bildrestauration                 | Entfernung von Rauschen, Kratzern oder Artefakten in alten oder beschädigten Bildern. |
|                                      | Colorisierung                    | Umfärben von Schwarzweiß-Bildern mit realistischen Farben.                            |
| 🧩 **Ergänzung & Vervollständigung** | **Inpainting**                       | Auffüllen fehlender oder beschädigter Bildbereiche.                                   |
|                                      | **Bildvervollständigung**        | Vorhersage fehlender Bildbereiche basierend auf Kontext.                              |
|                                      | Hintergrundentfernung            | Trennung des Vordergrundobjekts vom Hintergrund.                                      |
| 📚 **Informationsgewinnung**         | Texterkennung (OCR)              | Erkennung und Extraktion von Text aus Bildern.                                        |
|                                      | Schrifterkennung                 | Identifikation von Schriftarten oder Handschriften.                                   |
|                                      | **Bildbeschreibung (Captioning)**    | Automatische Erzeugung eines beschreibenden Textes zu einem Bild.                     |
|                                      | visuelle Fragebeantwortung (VQA) | Beantwortung von Fragen basierend auf einem Bildinhalt.                               |
| 🛡️ **Sicherheit & Strukturierung**  | Deepfake-Erkennung               | Analyse, ob ein Bild manipuliert oder synthetisch erzeugt wurde.                      |
|                                      | Gesichtsverpixelung              | Anonymisierung durch Unkenntlichmachung von Gesichtern.                               |
|                                      | Wasserzeichen-Erkennung          | Detektion sichtbarer oder unsichtbarer Wasserzeichen.                                 |
|                                      | Formatkonvertierung              | Umwandlung zwischen Bildformaten (z. B. PNG ↔ JPG).                                   |



# 3 | Bildgenerierung
---

Die Bildgenerierung mit Künstlicher Intelligenz (KI) ermöglicht es, aus einfachen Texteingaben oder bestehenden Bildern völlig neue, realistische oder kreative Grafiken zu erstellen. Moderne KI-Modelle wie GPT-Image setzen dabei auf fortschrittliche Algorithmen, die Texte präzise interpretieren und in beeindruckende visuelle Darstellungen umsetzen. Diese Technologie findet Anwendung in vielen Bereichen – von Design und Marketing über Kunst bis hin zur Bildung – und eröffnet neue kreative Möglichkeiten für Einsteiger und Profis gleichermaßen.


<p><font color='darkblue' size="4">
💡 <b>Viz</b>
</font></p>


- [Merkmals-Filter](https://editor.p5js.org/ralf.bendig.rb/full/zLXqi5u6f)
- [Merkmals-Filter-Anwendung](https://editor.p5js.org/ralf.bendig.rb/full/Xi2uabjR9)
- [CNN Explainer](https://poloclub.github.io/cnn-explainer/)


<p><font color='black' size="5">
Prompt-Beispiele
</font></p>

Prompts aus: [Versäumte Bilder](https://bilderinstitut.de/versaeumte-bilder-bonn)

**Foto Elvira Fölzer** standing proud as a female archaeologist, looking directly in the camera, as a 70 year old, on an bexcavation side with antic shards and ruins, photography from 1920, leica style, film gain --s 250 --v 6.1


**Foto Maria von Linde**n standing proud with crossed arms, as a 70 year old female scientist, dressed like a man, with slicked back hair, looking like a man, wearing a lab cout, standing in a lecture hall in Bonn in front of students, teaching chemistry, leica style from 1920 --v 6.0 --s 250


**Foto Leah Goldberg** as a 30 year old female scientist, sitting proud behind a desk with books, writing, illuminated by a small lamp, leica style from 1933, film gain --s 250 --v 6.1

... oder einfach: Erstelle eine Bild von einem Prosche 911 auf einer Küstenstraße. Das Bild sollte foto-realistisch und schwarz-weiss sein.

**⚠️ Wichtig: Import-Namenskonflikte vermeiden**

<details>

In diesem Notebook werden zwei Module verwendet, die beide eine `Image`-Funktion exportieren:

1. **`PIL.Image`** - Zum Öffnen, Bearbeiten und Speichern von Bildern
2. **`IPython.display.Image`** - Zum Anzeigen von Bildern in Jupyter

**Problem ohne Aliasing:**
```python
from PIL import Image          # Image = PIL.Image
from IPython.display import Image  # ❌ Überschreibt PIL.Image!
```

**✅ Lösung: Aliasing verwenden**

Wir verwenden in diesem Notebook folgende Konvention:
```python
from PIL import Image as PILImage              # Bildmanipulation
from IPython.display import Image as IPImage   # Anzeige in Jupyter
```

**Alternative Lösungen:**
```python
# Option 2: Modulimport statt direkter Import
import PIL.Image
from IPython import display
# Nutzung: PIL.Image.open() und display.Image()

# Option 3: Nur PIL importieren, display() für Anzeige
from PIL import Image
from IPython.display import display
# Nutzung: Image.open() und display(image)
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  GPT Image 2</font> </br></p>

prompt = "Portrait of Elvira Fölzer, a 70-year-old female archaeologist standing proudly on an excavation site with ancient shards and ruins, photographed in 1920s Leica style, film grain, high detail."

response = client.images.generate(model=IMAGE_GENERATION, prompt=prompt, size="1024x1024", quality="high", n=1)
img_data = base64.b64decode(response.data[0].b64_json)
img = PILImage.open(BytesIO(img_data))

img.save("bild_gpt_image_2.png")
show_img(img)

**Erläuterung:**

`BytesIO` ist ein dateiähnliches Objekt, das Daten im Speicher statt auf der Festplatte hält. Es verhält sich wie eine Datei, aber alles passiert im Arbeitsspeicher.


# 4 | Bildklassifizierung
---

Die Bildklassifizierung ist eine grundlegende Aufgabe in der Computer Vision, bei der einem gesamten Bild eine einzige Kategorie (Klasse) zugewiesen wird.

**Wie funktioniert Bildklassifizierung?**
+ Eingabe: Ein einzelnes Bild wird als Eingabe verwendet.
+ Merkmalsextraktion: Ein neuronales Netzwerk analysiert das Bild und extrahiert relevante Merkmale (z. B. Kanten, Farben, Formen, Muster).
+ Klassifizierung: Das Modell ordnet das Bild einer vordefinierten Kategorie zu, z. B. "Hund", "Katze" oder "Auto".
+ Ausgabe: Eine einzige Klasse (Label) mit einer Wahrscheinlichkeitsbewertung wird zurückgegeben.


Beispiel für Bildklassifizierung
Stelle dir vor, du hast ein Bild von einem Hund. Ein Bildklassifizierungsmodell verarbeitet das Bild und gibt die Kategorie "Hund" mit einer bestimmten Wahrscheinlichkeit (z. B. 95 %) zurück.

+ Eingabebild: Ein Bild eines Hundes
+ Modell-Ausgabe: "Hund" (95%)

Falls das Bild eine Katze zeigt, gibt das Modell möglicherweise "Katze" (90%) als Ergebnis zurück.

**Einschränkungen der Bildklassifizierung**     
Das Modell kann nur eine Klasse pro Bild erkennen, auch wenn mehrere Objekte im Bild vorhanden sind.
Es gibt keine Information über die Position oder Anzahl der Objekte im Bild.
Anwendungsfälle für Bildklassifizierung
+ Erkennung von medizinischen Anomalien (z. B. Klassifikation von Röntgenbildern)
+ Identifikation von Pflanzen oder Tieren anhand von Bildern
+ Sentiment-Analyse anhand von Gesichtsmerkmalen

**Bekannte Modelle für Bildklassifizierung**    
+ CNNs (Convolutional Neural Networks) wie ResNet, VGG, EfficientNet
+ Pretrained Modelle: MobileNet, Inception, AlexNet

Das Modell **google/vit-base-patch16-224** ist ein Vision Transformer (ViT) Modell, das von Google entwickelt wurde und sich auf Bildklassifizierungsaufgaben spezialisiert hat. Es gehört zur Familie der Transformer-Modelle, die ursprünglich für die Verarbeitung natürlicher Sprache (NLP) konzipiert wurden, aber hier auf visuelle Daten angewendet werden.

In [ ]:
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

classification_image_path = f"{IMAGE_DIR}/apfel.png"

display(IPImage(classification_image_path, width=500))

# Klassifizierung
classifier(classification_image_path)




Man kann allerdings auch gpt-5.4-mini mit einem entsprechenden Prompt hierzu einsetzen.

In [ ]:
image_path = f"{IMAGE_DIR}/apfel.png"
prompt = "Was ist auf diesem Bild abgebildet? Nennen nur das wesentliche Objekt."

result = analyze_image(image_path, prompt)
print(result)

# Bild anzeigen
display(IPImage(image_path, width=500))

**Hinweis zu Base64-Kodierung:**


Wann braucht man eine Base64-Kodierung?

✅ Ja - wenn binäre Daten (z. B. Bilder) in einem textbasierten Format übertragen werden müssen
    → z. B. JSON, Prompt, REST-Payload ohne Multipart

❌ Nein - wenn

+ eine öffentlich erreichbare URL existiert

+ ein binärer Upload (Multipart/Form-Data) unterstützt wird




Fazit:

* 📁 **Lokale Datei** → **Base64 + `data:`-URL**
* 🌐 **Online-URL** → **keine Base64 nötig**



# 5 | Inpainting
---

Inpainting ist eine Technik im Bereich der künstlichen Intelligenz und Bildverarbeitung, bei der bestimmte Teile eines Bildes automatisch ergänzt oder repariert werden. Die Technik wird eingesetzt, um:

+ Beschädigte Bereiche in Bildern zu rekonstruieren
+ Unerwünschte Objekte aus Bildern zu entfernen
+ Fehlende Teile in Bildern zu ergänzen

**Inpainting mit Maske:**   
Bei Inpainting mit Masken geht es darum, genau zu definieren, welche Bereiche eines Bildes rekonstruiert werden sollen. Die Maske ist dabei ein Schlüsselelement, das dem Algorithmus mitteilt, wo er arbeiten soll. Eine Maske ist ein Binärbild (Schwarz-Weiß-Bild), das die gleichen Dimensionen wie das Originalbild hat:

+ Weiße Bereiche (255): Zeigen an, welche Teile des Bildes rekonstruiert/gefüllt werden sollen
+ Schwarze Bereiche (0): Zeigen an, welche Teile des Originalbildes erhalten bleiben sollen

In [ ]:
prompt = "Ersetze den Kopf durch den Kopf eines alten Mannes. Behalte Licht, Schatten und Stil des Originals."
image_path = f"{IMAGE_DIR}/hedra_cyborg.png"
mask_path = f"{IMAGE_DIR}/hedra_cyborg_mask_image.png"

with open(image_path, "rb") as img_f, open(mask_path, "rb") as mask_f:
    response = client.images.edit(
        model=IMAGE_GENERATION,
        image=img_f,
        mask=mask_f,
        prompt=prompt,
        size="1024x1024",
        output_format="png"
    )

img_data = base64.b64decode(response.data[0].b64_json)
img = PILImage.open(BytesIO(img_data))
show_img(img)

# 6 | Outpainting
---

Outpainting ist eine Technik der Bildbearbeitung und KI-gestützten Bildgenerierung, bei der ein vorhandenes Bild über seine ursprünglichen Grenzen hinaus erweitert wird. Dabei werden neue Bildinhalte an den Rändern ergänzt, die stilistisch und thematisch zum Original passen. Outpainting wird häufig verwendet, um Bilder zu vergrößern, Hintergründe zu erweitern oder kreative Kompositionen zu schaffen. Moderne Methoden basieren oft auf neuronalen Netzwerken wie Stable Diffusion, die mithilfe von Textbeschreibungen (Prompts) realistische und kohärente Bildbereiche hinzufügen können. So entsteht ein nahtlos erweitertes Bild, das über das ursprüngliche Motiv hinausgeht.

In [ ]:
from PIL import ImageDraw  # nur hier benötigt (Outpainting-Maske)

# 1) Ausgangsbild laden
orig = PILImage.open(f"{IMAGE_DIR}/hedra_cyborg.png").convert("RGBA")

# Zielgröße für Outpainting (Querformat, ähnlich 16:9)
api_size = (1536, 1024)

# 2) Neues Canvas im Zielverhältnis erzeugen
canvas = PILImage.new("RGBA", api_size, (0, 0, 0, 0))
x_offset = (api_size[0] - orig.width) // 2
y_offset = (api_size[1] - orig.height) // 2
canvas.paste(orig, (x_offset, y_offset))
canvas.save("canvas.png")

# 3) Maske erzeugen (weiß = behalten, transparent = ergänzen)
mask = PILImage.new("RGBA", api_size, (0, 0, 0, 0))
draw = ImageDraw.Draw(mask)
draw.rectangle(
    [x_offset, y_offset, x_offset + orig.width, y_offset + orig.height],
    fill=(255, 255, 255, 255)
)
mask.save("mask.png")

# 4) API-Aufruf
prompt = (
    "Erweitere das Bild nach außen. "
    "Füge harmonisch passende Umgebung, Licht und Texturen hinzu, "
    "sodass der Stil und die Atmosphäre des Originals erhalten bleiben. "
    "Behalte Licht, Schatten und Stil des Originals."
)

with open("canvas.png", "rb") as img, open("mask.png", "rb") as m:
    response = client.images.edit(
        model=IMAGE_GENERATION,
        image=img,
        mask=m,
        prompt=prompt,
        size="1536x1024",
        quality="high",
        output_format="png"
    )

# Ausgangsbild
display(IPImage(f"{IMAGE_DIR}/hedra_cyborg.png", width=512))

# 5) Ausgabe speichern & anzeigen
img_data = base64.b64decode(response.data[0].b64_json)
img = PILImage.open(BytesIO(img_data))
filename = "outpaint_1536x1024.png"
img.save(filename)
print(f"Gespeichert: {filename}")
display(img.resize((768, 512)))


# 7 | Objekterkennung
---

Die Objekterkennung geht einen Schritt weiter als die Bildklassifizierung. Hier wird nicht nur bestimmt, welche Objekte in einem Bild vorhanden sind, sondern auch wo sie sich befinden.

**Wie funktioniert Objekterkennung?**   
+ Eingabe: Ein einzelnes Bild wird als Eingabe verwendet.
+ Merkmalsextraktion & Vorschläge für Regionen: Das Modell analysiert das Bild und identifiziert potenzielle Bereiche, in denen sich Objekte befinden könnten.
+ Klassifizierung & Lokalisierung: Jedes erkannte Objekt wird einer bestimmten Klasse zugeordnet, und die Position wird durch eine Bounding Box (ein rechteckiger Bereich um das Objekt) beschrieben.
+ Ausgabe: Eine Liste mit allen erkannten Objekten, ihrer Klasse und ihrer Position im Bild wird zurückgegeben.


**Beispiel für Objekterkennung**    
Stelle dir vor, du hast ein Bild mit einem Hund, einer Katze und einem Auto. Ein Objekterkennungsmodell kann alle drei Objekte gleichzeitig identifizieren und ihre Positionen angeben.

+ Eingabebild: Ein Bild mit einem Hund, einer Katze und einem Auto
+ Modell-Ausgabe:
>Hund (95%) - Position: (x1, y1, x2, y2)   
Katze (90%) - Position: (x3, y3, x4, y4)   
Auto (99%) - Position: (x5, y5, x6, y6)   

Hierbei sind (x1, y1, x2, y2) die Koordinaten der Bounding Box für den Hund, (x3, y3, x4, y4) für die Katze usw.

**Einschränkungen der Objekterkennung**   
Aufwendigere Berechnungen im Vergleich zur einfachen Bildklassifizierung.
Schwieriger zu trainieren, da genaue Positionen für Trainingsdaten erforderlich sind.
Anwendungsfälle für Objekterkennung
+ Autonomes Fahren (Erkennung von Fußgängern, Verkehrsschildern)
+ Überwachungssysteme (Erkennen von Eindringlingen oder gefährlichen Objekten)
+ Analyse von Bildern im Einzelhandel (z. B. automatische Produktzählung in Regalen)

**Bekannte Modelle für Objekterkennung**   
+ YOLO (You Only Look Once) – sehr schnell, ideal für Echtzeit
+ Faster R-CNN – präzise, aber langsamer
+ SSD (Single Shot MultiBox Detector) – guter Kompromiss zwischen Geschwindigkeit und Genauigkeit
+ **DETR (Detection Transformer)** – Transformer-basiert, direkt über HuggingFace nutzbar

<p><font color='black' size="5">
DETR
</font></p>

**DETR (Detection Transformer)** ist ein moderner Objekterkennungs-Algorithmus von Facebook AI Research (2020), der als erster Ansatz Transformer-Architektur direkt für Objekterkennung einsetzt – ohne die Anchor-Boxen oder NMS-Nachverarbeitung (Non-Maximum Suppression), die CNN-basierte Ansätze wie YOLO benötigen.

**Funktionsweise**

Ein CNN-Backbone (ResNet) extrahiert visuelle Merkmale aus dem Bild. Diese werden von einem Transformer-Encoder-Decoder verarbeitet: Der Encoder modelliert globale Beziehungen zwischen allen Bildregionen, der Decoder erzeugt parallel eine feste Anzahl von Objekt-Vorhersagen. Hungarian-Matching ordnet Vorhersagen und echte Objekte eindeutig zu – Duplikate entstehen gar nicht erst.

**Vorteile**

1. **Kein NMS nötig**: Der Transformer löst Duplikate direkt durch das Matching-Verfahren.
2. **Globaler Kontext**: Self-Attention erfasst Beziehungen zwischen Objekten im gesamten Bild.
3. **HuggingFace-Integration**: Direkt über `pipeline("object-detection")` nutzbar – dasselbe Muster wie Bildklassifizierung in Kapitel 4.

**Einschränkungen**

- Langsamer beim Training als YOLO (konvergiert erst nach vielen Epochen).
- Für Echtzeitanwendungen mit hoher FPS-Anforderung weniger geeignet.

- [DETR auf HuggingFace](https://huggingface.co/facebook/detr-resnet-50)
- [DETR Paper (Facebook AI, 2020)](https://arxiv.org/abs/2005.12872)

Der Code verwendet DETR (`facebook/detr-resnet-50`) über die HuggingFace `transformers`-Pipeline. Das Bild wird mit `matplotlib.pyplot.imread()` als numpy-Array geladen (kein PIL nötig), Bounding Boxes und Labels werden interaktiv mit Plotly dargestellt — Zoom und Hover zeigen Klasse und Konfidenz direkt im Diagramm.

In [ ]:
# DETR-Modell laden (kein separates Paket nötig — transformers ist vorinstalliert)
detector = pipeline("object-detection", model="facebook/detr-resnet-50")

In [ ]:
import plotly.express as px
import matplotlib.pyplot as plt

detection_image_path = f"{IMAGE_DIR}/peoples.png"

# Bild als numpy-Array laden (kein PIL — matplotlib liest PNG via libpng direkt)
img_array = plt.imread(detection_image_path)

# Objekterkennung
results = detector(detection_image_path, threshold=0.5)
print(f"Erkannte Objekte: {len(results)}")

# Interaktive Visualisierung mit Plotly (Zoom, Hover)
fig = px.imshow(img_array)
for r in results:
    box = r["box"]
    fig.add_shape(
        type="rect",
        x0=box["xmin"], y0=box["ymin"],
        x1=box["xmax"], y1=box["ymax"],
        line=dict(color="red", width=2)
    )
    fig.add_annotation(
        x=box["xmin"], y=box["ymin"],
        text=f"{r['label']} {r['score']:.0%}",
        xanchor="left", yanchor="bottom",
        showarrow=False,
        font=dict(color="white", size=11),
        bgcolor="red", opacity=0.85
    )
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=900, height=600)
fig.show()

# 8 | Bildbeschreibung

Die Aufgabe Bildbeschreibung (Image2Text) beschreibt die automatische Generierung von Text aus Bildern mithilfe von Künstlicher Intelligenz (KI). Dabei werden neuronale Netze, insbesondere tiefgehende Modelle in Kombination mit Transformern, genutzt, um visuelle Inhalte zu analysieren und zu beschreiben.

**Anwendungsfälle**    
+ Bildbeschreibung (Image Captioning): Generierung von Textbeschreibungen zu Bildern (z. B. für barrierefreie Webseiten oder Archivierung).
+ Optische Zeichenerkennung (OCR): Extraktion von Text aus Bildern oder gescannten Dokumenten.
+ Visuelle Frage-Antwort-Systeme (Visual Question Answering, VQA): Beantwortung von Fragen zu Bildinhalten.
+ Content-Moderation: Automatische Identifikation und Beschreibung von problematischen oder sensiblen Bildinhalten.

Der Code lädt ein Bild in Google Colab hoch, kodiert es in Base64 und sendet es zusammen mit einer Frage ("Was ist auf dem Bild?") an das OpenAI-Modell "gpt-5.4-mini". Die Antwort des Modells (eine detaillierte Beschreibung im Markdown-Format) wird angezeigt, ebenso wie das hochgeladene Bild selbst (mit einer maximalen Breite von 500 Pixeln).


In [ ]:
description_image_path = f"{IMAGE_DIR}/peoples.png"
prompt = "Was ist auf dem Bild? Erstelle eine ausführliche Beschreibung im Markdown-Format. Prüfe die Bildinhalte sorgfältig!"

result = analyze_image(description_image_path, prompt)

display(IPImage(description_image_path, width=750))
mprint(result)


# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei einer Aufgabe, bei der man festhängt, kann zum Beispiel Gemini in Google Colab helfen: Fehlermeldungen besser verstehen, Ideen für Teilschritte entwickeln oder Code-Varianten prüfen.
> Wichtig ist: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Coverbild LLM-Buch
</font></p>

In einem früheren Modul wurde ein LLM-basierten Buchgenerator erstellt, der die Erstellung eines Buches von Anfang bis Ende automatisiert. Dieser Prozess umfasste die Erstellung einer Zusammenfassung, die Gliederung des Inhaltsverzeichnisses und das iterative Schreiben der Kapitel in ein strukturiertes Markdown-Dokument. Jetzt wird diese Arbeit erweitert: Erstellen Sie mit GPT Image 2 ein Coverbild für das Buch, das den textbasierten Inhalt ergänzt.

**✅ Erledigt wenn:** Das generierte Bild passt visuell zum Buchtitel und zur Synopsis aus M05 — kein generisches Buchcover ohne inhaltlichen Bezug.

In [ ]:
# Starter-Template: Coverbild mit GPT Image 2 generieren
# Startpunkt: Bildgenerierungs-Beispiele aus Kapitel 2

# 1. Buchtitel + Synopsis aus M05 laden oder eintragen
buchtitel = "..."   # aus M05
synopsis  = "..."   # aus M05, 2–3 Sätze

# 2. Prompt für Bildgenerierung formulieren
bild_prompt = f"Buchcover für '{buchtitel}': {synopsis}. Realistisch, detailreich."

# 3. Bildgenerierungsmodell aufrufen
# from openai import OpenAI
# client = OpenAI()
# response = client.images.generate(model=IMAGE_GENERATION, prompt=bild_prompt, size='1024x1024')

# 4. Bild anzeigen
# ...

# B | Dokumente zum Weiterlesen
---



📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Multimodal: Bild](https://ralf-42.github.io/GenAI/09-multimodal/multimodal-bild.html)
- [Embeddings](https://ralf-42.github.io/GenAI/03-grundlagen/embeddings.html)